# Maia Lite 0.3.0 — Corpus v8 Dataloader & Deterministic Resume v1

**Purpose:** inspect frozen inventory and tokenizer; build a deterministic, seekable block sampler; verify restart without repeating/skipping blocks; stage real data only after documented source-specific authorization. **Default: metadata-only and synthetic validation.** Does not alter original SQLite, shards, corpus, or the synthetic model checkpoints.

**Important:** corpus data rights are still pending; the real-data loader refuses access unless an independently reviewed approval manifest is supplied. This notebook does **not** create approvals or start production training. A model-only checkpoint is insufficient for an end-to-end production resume; training checkpoints must also save the sampler state and dataset fingerprint.

In [ ]:
import os, json, math, hashlib, sqlite3, random, struct, time
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass
PROJECT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
OUT = PROJECT/'corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full'
RECOVERY = OUT/'recovery_audit_v4'
CONFIG = OUT/'run_config.json'
DB = RECOVERY/'state_operational_v5.sqlite3'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
INV_SHA_EXPECTED = 'ad19d1d98e2cf5b91e650f343e6430bbde34b549b5fbc352b47682a91d7ab562'
BLOCK = 2048
assert CONFIG.is_file() and DB.is_file(), 'Production config or operational v5 SQLite not found'
config=json.loads(CONFIG.read_text(encoding='utf-8'))
assert config.get('block_size') == BLOCK
inventory=config['selected_inventory']
inv_sha=hashlib.sha256(json.dumps(inventory,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
assert inv_sha==INV_SHA_EXPECTED, 'Inventory fingerprint changed'
expected=config.get('tokenizer_file_sha256',{})
assert expected, 'No tokenizer fingerprints in run config'
for name,digest in expected.items():
    file=TOKENIZER_DIR/name
    assert file.is_file() and hashlib.sha256(file.read_bytes()).hexdigest()==digest, f'Tokenizer mismatch: {name}'
print('Inventory SHA verified:', inv_sha)
print('Tokenizer files verified:',len(expected))
print('SQLite:',DB)

## 1. Inspect shard catalog without reading token payloads

The `shard` table is the authority for completed token blocks. Pending tails are **not** full blocks and are excluded. Paths and SHA-256 are retained. No source is eligible merely because it has completed shards.

In [ ]:
uri=DB.resolve().as_uri()+'?mode=ro'
con=sqlite3.connect(uri,uri=True,timeout=60)
try:
    assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
    schema={r[1] for r in con.execute('PRAGMA table_info(shard)')}
    assert {'source','split','shard_index','path','blocks','sha256'}<=schema
    shard_rows=con.execute('SELECT source,split,shard_index,path,blocks,sha256 FROM shard ORDER BY source,split,shard_index').fetchall()
    completed=con.execute('SELECT source,COUNT(*) FROM completed_chunk GROUP BY source').fetchall()
finally:
    con.close()
assert shard_rows, 'No committed shards'
CATALOG=[]
for source,split,idx,path,blocks,sha in shard_rows:
    assert split in ('train','validation','test') and blocks>0
    assert len(sha)==64 and all(c in '0123456789abcdef' for c in sha.lower())
    CATALOG.append(dict(source=source,split=split,index=idx,path=path,blocks=blocks,sha256=sha))
summary=defaultdict(int)
for s in CATALOG:summary[(s['source'],s['split'])]+=s['blocks']
for (source,split),blocks in sorted(summary.items()):
    print(f'{source:17} {split:11} {blocks:>12,} blocks | {blocks*BLOCK:>15,} tokens')
print('Completed chunks:',dict(completed))
print('Committed shards:',len(CATALOG))

## 2. Frozen authorization gate

A reviewer must independently create a JSON manifest with `inventory_sha256`, `tokenizer_file_sha256`, and `source_decisions`. Each source decision needs `training_approved: true`, `reviewer`, `reviewed_at_utc`, `evidence`, and `scope: pretraining`. The manifest is **not** generated by this notebook. All six sources remain excluded by default. This is a technical enforcement format, not a substitute for legal review.

In [ ]:
APPROVAL_FILE = RECOVERY/'production_resume_v2/license_approval.json'
# This code does not interpret the existence of an unrelated approval file as authorization.
# Only an explicit, schema-validated per-source manifest is accepted.
REVIEWED_SOURCE_MANIFEST = PROJECT/'compliance/maia_v8_source_training_approvals.json'
SOURCES=sorted({s['source'] for s in CATALOG})

def approved_sources():
    if not REVIEWED_SOURCE_MANIFEST.is_file():
        return set()
    a=json.loads(REVIEWED_SOURCE_MANIFEST.read_text(encoding='utf-8'))
    assert a.get('inventory_sha256')==inv_sha, 'Approval inventory SHA mismatch'
    assert a.get('tokenizer_file_sha256')==expected, 'Approval tokenizer SHA mismatch'
    decisions=a.get('source_decisions',{})
    assert set(decisions)<=set(SOURCES), 'Unknown approved source'
    approved=set()
    for source,entry in decisions.items():
        if entry.get('training_approved') is True:
            for key in ('reviewer','reviewed_at_utc','evidence'):
                assert entry.get(key), f'Incomplete rights review: {source}/{key}'
            assert entry.get('scope')=='pretraining', f'Wrong rights scope: {source}'
            approved.add(source)
    return approved

APPROVED=approved_sources()
print('Sources:',SOURCES)
print('Sources explicitly approved:',sorted(APPROVED))
print('Production data eligible:',bool(APPROVED))

## 3. Deterministic block sampler

Use a stable affine permutation per epoch, indexed directly by global block position. It is a bijection over the approved training blocks, so within one epoch it never repeats or skips blocks. Its position is an integer saved in a checkpoint. Multi-worker prefetching does not change the committed position; only increment it after the optimizer update completes. Source mixture follows the materialized corpus distribution (no invented weights). Validation/test blocks are excluded.

In [ ]:
import bisect
class SeekableBlockSampler:
    def __init__(self,shards,seed=20261105):
        self.shards=sorted([s for s in shards if s['split']=='train'],key=lambda s:(s['source'],s['index']))
        self.ends=[];total=0
        for s in self.shards:
            total+=int(s['blocks']);self.ends.append(total)
        self.total=total;self.seed=int(seed)
        assert self.total>0
    def _params(self,epoch):
        n=self.total
        rng=random.Random(f'{self.seed}:{epoch}:{n}')
        a=rng.randrange(1,n) if n>1 else 1
        while math.gcd(a,n)!=1:a=(a+1)%n or 1
        b=rng.randrange(n)
        return a,b
    def at(self,position):
        assert isinstance(position,int) and position>=0
        epoch,within=divmod(position,self.total)
        a,b=self._params(epoch)
        block=(a*within+b)%self.total
        shard_index=bisect.bisect_right(self.ends,block)
        base=0 if shard_index==0 else self.ends[shard_index-1]
        s=self.shards[shard_index]
        return {'position':position,'epoch':epoch,'source':s['source'],
                'path':s['path'],'offset_blocks':block-base,'sha256':s['sha256']}
    def fingerprint(self):
        canonical=[(s['source'],s['split'],s['index'],s['path'],s['blocks'],s['sha256']) for s in self.shards]
        return hashlib.sha256(json.dumps(canonical,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()

# Synthetic-only surrogate: validate algorithm without accessing restricted data.
synthetic=[dict(source='synthetic_pt',split='train',index=0,path='synthetic://pt',blocks=17,sha256='0'*64),
           dict(source='synthetic_en',split='train',index=0,path='synthetic://en',blocks=23,sha256='1'*64),
           dict(source='synthetic_es',split='train',index=0,path='synthetic://es',blocks=19,sha256='2'*64),
           dict(source='synthetic_pt',split='validation',index=0,path='synthetic://val',blocks=7,sha256='3'*64)]
sampler=SeekableBlockSampler(synthetic)
first=[sampler.at(i) for i in range(sampler.total)]
assert len({(r['path'],r['offset_blocks']) for r in first})==sampler.total
assert all(r['path']!='synthetic://val' for r in first)
assert sampler.at(sampler.total)['epoch']==1
print('SYNTHETIC SAMPLER PASS:',sampler.total,'unique training blocks per epoch')

## 4. Simulated interruption and exact resume

A step commits `microbatch × accumulation` block positions. Save `next_position` **after** the optimizer step; if a crash happens before atomic checkpoint commit, restart from the previous checkpoint and redo its subsequent steps, not from a partially committed cursor. This verifies the sampler, not full bitwise equivalence of a GPU training run.

In [ ]:
MICROBATCH=8; ACCUM=4; BLOCKS_PER_UPDATE=MICROBATCH*ACCUM
TEST_STEPS=10
baseline=[sampler.at(i) for i in range(TEST_STEPS*BLOCKS_PER_UPDATE)]
# emulate checkpoint after update 5
resume_state={'completed_updates':5,'next_position':5*BLOCKS_PER_UPDATE,
              'sampler_fingerprint':sampler.fingerprint(),'sampler_seed':sampler.seed}
assert resume_state['sampler_fingerprint']==sampler.fingerprint()
restored=[sampler.at(i) for i in range(resume_state['next_position'],TEST_STEPS*BLOCKS_PER_UPDATE)]
assert baseline[5*BLOCKS_PER_UPDATE:]==restored
assert baseline[:5*BLOCKS_PER_UPDATE]+restored==baseline
assert all(baseline[i]['position']==i for i in range(len(baseline)))
print('RESUME SAMPLER VERIFIED: updates 5 -> 10')
print('Next block position:',TEST_STEPS*BLOCKS_PER_UPDATE)
print('Blocks per update:',BLOCKS_PER_UPDATE)

## 5. Real-shard reader — gated, read-only, SHA checked

The following definitions do not open any corpus payloads. Once source-specific approval exists, an explicit call can verify an approved shard SHA and read one fixed-size little-endian uint32 block. Do not execute the gated test while approvals are pending. **Caution:** the original corpus stores exactly 2048 token IDs per block; causal next-token targets must use a documented cross-block boundary policy (e.g., concatenate adjacent blocks only if sequence provenance permits, or mask the last position). Never silently roll targets within a block.

In [ ]:
def verified_shard_path(entry):
    assert entry['source'] in approved_sources(), 'Source not approved for training'
    p=Path(entry['path'])
    assert p.is_file(), f'Missing shard: {p}'
    assert p.stat().st_size == entry['blocks']*BLOCK*4, 'Shard length mismatch'
    h=hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    assert h.hexdigest()==entry['sha256'], 'Shard SHA-256 mismatch'
    return p

def read_verified_block(entry,offset_blocks):
    assert 0<=offset_blocks<entry['blocks']
    p=verified_shard_path(entry)
    with p.open('rb') as f:
        f.seek(offset_blocks*BLOCK*4)
        raw=f.read(BLOCK*4)
    assert len(raw)==BLOCK*4
    ids=np.frombuffer(raw,dtype='<u4').copy()
    assert ids.size==BLOCK and ids.max()<50257, 'Invalid token ID'
    return ids

RUN_APPROVED_SHARD_TEST=False
if RUN_APPROVED_SHARD_TEST:
    assert APPROVED, 'No source-level approval manifest'
    real=[s for s in CATALOG if s['split']=='train' and s['source'] in APPROVED]
    assert real, 'No approved train shards'
    ids=read_verified_block(real[0],0)
    print('Approved shard test OK:',real[0]['source'],'tokens:',len(ids))
else:
    print('Real shard access DISABLED; definitions loaded only')

## 6. Production handoff checklist

- **Do not** use the synthetic checkpoints (`checkpoint_0000010.pt`) as production weights. Start from a new recorded random seed.
- Validate full train/validation/test manifest, tokenizer special IDs, and document-boundary loss masking. Existing 2048-token blocks cannot automatically provide 2048 valid next-token labels without an extra token.
- Before using a shard, validate its recorded SHA-256; caching a verified shard locally is preferable to rehashing on every block.
- Save in a *single atomic checkpoint*: model, AdamW, scheduler, CPU/CUDA RNG, run identity, `completed_updates`, `next_position`, approved source set, inventory SHA, tokenizer hashes, shard-catalog fingerprint, and train/validation split identity. Verify restored optimizer state and perform uninterrupted-versus-resumed comparison.
- Use a separate production checkpoint directory. Confirm sufficient Google Drive storage and implement retention of recent complete checkpoints.
- Chinchilla target: 10,140,528,640 token exposures. With 65,536 tokens/update, `ceil` gives 154,733 updates (slightly exceeding the target). The available 6.94B materialized training tokens are **not** an authorized training budget.
- **Approval pending:** This notebook cannot authorize or begin real-data training. Review all source-specific rights and document decisions before enabling the real reader.